# R Regression, Correlation, and Diagnostics

**Tier 0 -- Computational Foundations | Module 8, Notebook 2**

---

## Overview

This notebook covers parametric inference, normality testing, correlation analysis, contingency table methods, linear regression, ANOVA, and simulation-based demonstrations of the Law of Large Numbers and Central Limit Theorem — all implemented in R.

### Learning Objectives

By the end of this notebook you will be able to:
- Construct Student-t, chi-squared, and normal asymptotic confidence intervals
- Run paired and unpaired t-tests and test variance equality with `var.test`
- Assess normality using Shapiro-Wilk, Pearson chi-squared, and Q-Q plots
- Compute Pearson, Spearman, and Kendall correlation with Fisher z-transform CIs
- Build contingency tables and apply chi-squared independence and homogeneity tests
- Fit simple and multiple linear regression models, evaluate residual diagnostics, and select polynomial degree
- Compare ANOVA and Kruskal-Wallis on the same dataset
- Demonstrate the Law of Large Numbers and Central Limit Theorem by simulation

## Why this notebook matters

Regression and correlation are the workhorses of quantitative biology. You use linear regression to model dose-response relationships, find genes correlated with clinical outcomes, and control for confounders (e.g., batch effects, age, sex). Diagnostic plots tell you whether your model's assumptions are met — a regression that passes diagnostics is trustworthy; one that fails is misleading. This notebook builds the practical R skills for fitting, interpreting, and validating regression models on biological data.


## How to use this notebook
1. Run cells top-to-bottom — datasets are simulated at the start of each section and reused in subsequent cells.
2. When you run a `lm()` call, always run the diagnostic plots (`plot(model)`) immediately after. A model without diagnostics is incomplete.
3. For each regression output, locate the coefficient estimates, standard errors, and p-values — these are the quantities you will report in a paper.
4. The multiple regression section introduces confounders: a concept critical to valid bioinformatics analysis.


## Common stumbling points

- **R-squared is not the whole story**: A high R² does not mean the model is correct or that the relationship is causal. It just means the model explains variance. Check diagnostics and think about biology.
- **Residuals vs. Fitted plot**: The single most important diagnostic. A clear pattern in residuals (curve, funnel shape) means your model is misspecified or assumptions are violated. Random scatter around zero is what you want.
- **Pearson vs. Spearman correlation**: Pearson measures linear association and requires roughly normal data. Spearman measures monotonic association using ranks and is appropriate for skewed data (read counts, survival times) or when you just need to know "do they go up and down together."
- **Confounders in multiple regression**: Adding a confounder variable to a model changes the coefficients of all other variables. This is not a bug — it is the model correctly partitioning variance. A coefficient's meaning is "the effect of this variable, *holding all other variables constant*."


---

## 1. Confidence Intervals

A **confidence interval** (CI) gives a range of plausible values for a population parameter based on sample data. A 95% CI means: if we repeated the experiment many times and computed a CI each time, 95% of those intervals would contain the true parameter.

In R, the correct CI depends on what you know:
- **Student-t CI**: sigma (population SD) is unknown — use the t-distribution with n-1 degrees of freedom. This is the most common case.
- **Chi-squared CI**: for a variance (sigma²) when data is normal.
- **Normal asymptotic CI**: for counts (Poisson mean) or proportions (binomial parameter) with large n.


In [ ]:
# Load IQ data
iq_data  <- read.table("../Assets/data/r_statistics/IQ.txt",
                       header = TRUE, sep = " ", dec = ".")
iq_data  <- iq_data[[1]]

n        <- length(iq_data)
iq_mean  <- mean(iq_data)           # sample mean
iq_s     <- sd(iq_data)             # corrected sample standard deviation
alpha    <- 0.10

cat("n =", n, "  mean =", round(iq_mean, 2), "  s =", round(iq_s, 2), "\n")

In [ ]:
# Student-t confidence interval for the mean (sigma unknown)
student_q <- qt(1 - alpha/2, df = n - 1)

ci_left  <- iq_mean - student_q * iq_s / sqrt(n)
ci_right <- iq_mean + student_q * iq_s / sqrt(n)
cat(sprintf("90%% two-sided CI for mean: [%.2f, %.2f]\n", ci_left, ci_right))

# One-sided (right) CI: mean > lower bound
ci_right_one <- iq_mean + qt(1 - alpha, df = n - 1) * iq_s / sqrt(n)
cat(sprintf("90%% right-sided CI:  mean > %.2f\n",
            iq_mean - qt(1 - alpha, df = n - 1) * iq_s / sqrt(n)))

In [ ]:
# t-test also returns a CI:
t.test(iq_data, alternative = "greater", mu = 110, conf.level = 1 - alpha)

# Manual t-statistic for H0: mu = 110
t_stat   <- (iq_mean - 110) * sqrt(n) / iq_s
iq_crit  <- qt(1 - alpha, df = n - 1)
cat(sprintf("t = %.3f  critical value = %.3f  reject H0: %s\n",
            t_stat, iq_crit, t_stat > iq_crit))

In [ ]:
# Chi-squared confidence interval for variance
# Example: n=30 product quality measurements, sample variance = 0.3
n       <- 30
var_obs <- 0.3    # this is s^2, not sd
alpha   <- 0.05

# Test H0: sigma^2 = 0.2 against H1: sigma^2 > 0.2
T_stat  <- (n - 1) * var_obs / 0.2
crit    <- qchisq(1 - alpha, df = n - 1)
cat(sprintf("chi^2 statistic = %.2f  critical value = %.2f  reject H0: %s\n",
            T_stat, crit, T_stat > crit))

# Two-sided CI for sigma^2
ci_var_left  <- (n - 1) * var_obs / qchisq(1 - alpha/2, df = n - 1)
ci_var_right <- (n - 1) * var_obs / qchisq(alpha/2,     df = n - 1)
cat(sprintf("95%% CI for sigma^2: [%.4f, %.4f]\n", ci_var_left, ci_var_right))

In [ ]:
# Normal asymptotic CI for a count parameter (Poisson mean)
# Example: store receives on average 256 customers; n=100 days observed
n         <- 100
mean_shop <- 256
alpha     <- 0.01

norm_q    <- qnorm(1 - alpha/2)
ci_left   <- mean_shop - norm_q * sqrt(mean_shop) / sqrt(n)
ci_right  <- mean_shop + norm_q * sqrt(mean_shop) / sqrt(n)
cat(sprintf("99%% CI for Poisson mean: [%.2f, %.2f]\n", ci_left, ci_right))

In [ ]:
# Normal asymptotic CI for proportion (binomial parameter)
# Example: 180 of 300 patients improved
n      <- 300
scs    <- 180
alpha  <- 0.05
p_hat  <- scs / n

norm_q <- qnorm(1 - alpha/2)
ci_left  <- p_hat - norm_q * sqrt(p_hat * (1 - p_hat) / n)
ci_right <- p_hat + norm_q * sqrt(p_hat * (1 - p_hat) / n)
cat(sprintf("95%% CI for proportion: [%.4f, %.4f]\n", ci_left, ci_right))

---

## 2. Parametric t-Tests

When the data can be assumed normally distributed, the **t-test** is more powerful than nonparametric alternatives. Use `var.test` to check variance equality before choosing between equal- and unequal-variance t-tests.

In [ ]:
# Paired t-test: weight before and after diet intervention
# (same data as Notebook 1 — comparing parametric and nonparametric results)
weight_data <- read.table("../Assets/data/r_statistics/Weight.txt",
                          header = TRUE, sep = " ", dec = ".")
before <- weight_data[[1]]
after  <- weight_data[[2]]

# H1: weight decreases after diet (after < before)
t.test(after, before, alternative = "less", mu = 0, paired = TRUE)

In [ ]:
# Independent two-sample t-test: compare two towns
towns_data <- read.table("../Assets/data/r_statistics/town.txt",
                         header = TRUE, sep = " ", dec = ".")
skov    <- towns_data[[1]]
kastrul <- towns_data[[2]]

# Step 1: Test equality of variances (F-test)
var.test(skov, kastrul, ratio = 1, alternative = "two.sided")

In [ ]:
# Step 2: If variances are unequal, use var.equal = FALSE (Welch's t-test)
t.test(skov, kastrul,
       alternative = "greater", mu = 0,
       paired = FALSE, var.equal = FALSE)

In [ ]:
# Comparison: nonparametric Wilcoxon rank-sum on the same data
library(exactRankTests)
wilcox.exact(skov, kastrul, paired = FALSE, alternative = "greater", exact = TRUE)

---

## 3. Normality Testing

Before applying parametric tests, check whether the data are consistent with a normal distribution.

| Method | Function | Notes |
|--------|----------|-------|
| Histogram | `hist` | Visual only |
| Q-Q plot | `qqnorm` + `qqline` | Points should fall on the line |
| Shapiro-Wilk | `shapiro.test` | Most powerful for $n \leq 5000$ |
| Pearson chi-squared GoF | `pearson.test` (nortest) | Requires choosing bin count |

A **low p-value** from `shapiro.test` is evidence *against* normality.

In [ ]:
# Load weather data: temperature measurements
weather_data <- read.table("../Assets/data/r_statistics/weather.txt",
                           header = TRUE, sep = "\t", dec = ".")

# Extract summer temperatures (months 6-8)
temp_summer <- as.numeric(weather_data$Temp[
    weather_data$Month >= 6 & weather_data$Month <= 8
])

cat("Summer temperature observations: n =", length(temp_summer), "\n")
cat("Mean:", round(mean(temp_summer), 1), " SD:", round(sd(temp_summer), 1), "\n")

In [ ]:
# Visual normality checks
par(mfrow = c(1, 2))

hist(temp_summer, breaks = 15,
     main = "Summer Temperatures",
     xlab = "Temperature (°C)", col = "lightblue")

qqnorm(temp_summer, main = "Q-Q Plot: Summer Temperatures")
qqline(temp_summer, col = "red", lwd = 2)

par(mfrow = c(1, 1))

In [ ]:
# Shapiro-Wilk test
shapiro.test(temp_summer)

In [ ]:
# Pearson chi-squared goodness-of-fit test for normality
# install.packages("nortest")
library(nortest)

# adjust=FALSE: n-1 degrees of freedom (conservative)
# adjust=TRUE:  n-3 degrees of freedom (corrects for estimated mean and sd)
# True p-value lies between the two
pearson.test(temp_summer, adjust = FALSE)
pearson.test(temp_summer, adjust = TRUE)

In [ ]:
# Second dataset: lake depth measurements
lake_data   <- read.table("../Assets/data/r_statistics/Lake.txt",
                          header = TRUE, sep = " ", dec = ".")
lake_column <- lake_data[[1]]

par(mfrow = c(1, 2))
hist(lake_column, breaks = 12, main = "Lake Depths",
     xlab = "Depth (m)", col = "lightgreen")
qqnorm(lake_column, main = "Q-Q Plot: Lake Depths")
qqline(lake_column, col = "red", lwd = 2)
par(mfrow = c(1, 1))

shapiro.test(lake_column)
pearson.test(lake_column, adjust = TRUE)

---

## 4. Correlation Analysis

### Pearson Correlation

The Pearson coefficient $r$ measures linear association. It assumes both variables are normally distributed (for inference). The `cor.test` function tests $H_0: \rho = 0$ and returns a confidence interval via Fisher's z-transform.

### Spearman and Kendall Rank Correlations

Rank-based correlations detect **monotone** (not just linear) relationships and do not require normality.

In [ ]:
# Load country data: two continuous variables per country
corr_data <- read.table("../Assets/data/r_statistics/country.txt",
                        header = TRUE, sep = ";", dec = ",")

# Note: column 1 contains row numbers — skip it
x1 <- corr_data[[2]]
y1 <- corr_data[[3]]

plot(x1, y1,
     xlab = colnames(corr_data)[2],
     ylab = colnames(corr_data)[3],
     main = "Scatter Plot", pch = 20)

In [ ]:
# Check normality before choosing correlation method
shapiro.test(x1)
shapiro.test(y1)

In [ ]:
# Pearson correlation with 95% CI
cor.test(x1, y1, alternative = "two.sided", method = "pearson", conf.level = 0.95)

In [ ]:
# Spearman rank correlation
cor.test(x1, y1, alternative = "two.sided", method = "spearman")

# Kendall tau
cor.test(x1, y1, alternative = "two.sided", method = "kendall")

### 4.1 Fisher z-Transform CI for Pearson Correlation

The Fisher z-transform $z = \frac{1}{2}\ln\frac{1+r}{1-r}$ is approximately $N(\rho_z, 1/(n-3))$ for large $n$. This stabilizes the variance and allows direct CI construction.

In [ ]:
n  <- length(x1)

# Manual Pearson r
r  <- (mean(x1 * y1) - mean(x1) * mean(y1)) / (sd(x1) * sd(y1) * (n - 1) / n)
cat("Pearson r (manual):", round(r, 4), "\n")

# Fisher z-transform
z1       <- 0.5 * log((1 + r) / (1 - r))
alpha    <- 0.05
norm_q   <- qnorm(1 - alpha/2)

# CI bounds on the z scale
z1_left  <- z1 - norm_q / sqrt(n - 3)
z1_right <- z1 + norm_q / sqrt(n - 3)

# Back-transform to r scale
r_left   <- (exp(2 * z1_left)  - 1) / (exp(2 * z1_left)  + 1)
r_right  <- (exp(2 * z1_right) - 1) / (exp(2 * z1_right) + 1)

cat(sprintf("95%% CI for rho: [%.4f, %.4f]\n", r_left, r_right))

In [ ]:
# Demonstration: different relationship shapes affect Pearson vs Spearman

# Scenario A: Pearson picks up linear relationship well
x2 <- abs(x1) + rnorm(length(x1), 0, 0.03)
y2 <- abs(y1) + rnorm(length(y1), 0, 0.02)
cat("Pearson r (abs transformation):",
    round(cor.test(x2, y2, method="pearson")$estimate, 3), "\n")
cat("Spearman rho (abs transformation):",
    round(cor.test(x2, y2, method="spearman")$estimate, 3), "\n\n")

# Scenario B: Quadratic relationship — Spearman detects monotone dependence
x3 <- x1 + rnorm(length(x1), 0, 0.03)
y3 <- x1^2 + rnorm(length(x1), 0, 0.02)
cat("Pearson r (quadratic):",
    round(cor.test(x3, y3, method="pearson")$estimate, 3), "\n")
cat("Spearman rho (quadratic):",
    round(cor.test(x3, y3, method="spearman")$estimate, 3), "\n")

---

## 5. Chi-Squared Tests

### 5.1 Test of Independence

Tests whether two categorical variables are associated in a contingency table.

**Null hypothesis:** The two variables are independent.  
**Statistic:** $\chi^2 = \sum \frac{(O - E)^2}{E}$, where $E = \frac{\text{row total} \times \text{column total}}{n}$.  
**Degrees of freedom:** $(r-1)(c-1)$.

In [ ]:
# Manually build a 2x4 contingency table
# Response times (fast/slow) across 4 age groups
cont_table <- matrix(
    c(120, 124, 133, 106,
       97, 142, 129, 149),
    nrow = 2, ncol = 4, byrow = TRUE
)
colnames(cont_table) <- c("18-30", "31-45", "46-60", "60+")
rownames(cont_table) <- c("Fast", "Slow")
cont_table

# Chi-squared test of independence
chisq.test(cont_table)

In [ ]:
# Critical value (for manual comparison)
df    <- (2 - 1) * (4 - 1)
alpha <- 0.05
cat("Critical value (chi^2,", df, "df):", qchisq(1 - alpha, df), "\n")

In [ ]:
# Contingency table from student dataset (hair and eye color)
students_data  <- read.table("../Assets/data/r_statistics/students.txt",
                             header = TRUE, sep = "\t", dec = ".")

# xtabs builds a cross-tabulation from formula
cont_table_stu <- xtabs(Freq ~ Hair + Eye, data = students_data)
ftable(cont_table_stu)   # display in standard form

chisq.test(cont_table_stu)

### 5.2 Test of Homogeneity

Tests whether multiple populations share the same distribution of a categorical variable. Uses the same $\chi^2$ statistic but the sampling scheme differs (fixed row totals).

In [ ]:
# Two independent samples: success/failure counts
# Group 1: 60 successes out of 500 trials
# Group 2: 42 successes out of 600 trials
cont_hom <- matrix(c(60, 440,
                     42, 558),
                   nrow = 2, ncol = 2, byrow = TRUE)
colnames(cont_hom) <- c("Success", "Failure")
rownames(cont_hom) <- c("Group1", "Group2")

chisq.test(cont_hom)

In [ ]:
# Equivalent using prop.test (two-proportion z-test)
success <- c(60, 42)
total   <- c(500, 600)
prop.test(success, total, alternative = "two.sided")

In [ ]:
# Test whether blond hair proportions differ between male and female students
Blond_n  <- with(students_data, c(
    sum(Freq[Sex == "Male"   & Hair == "Blond"]),
    sum(Freq[Sex == "Female" & Hair == "Blond"])
))
Total_n  <- with(students_data, c(
    sum(Freq[Sex == "Male"]),
    sum(Freq[Sex == "Female"])
))

cat("Blond counts:", Blond_n, " Total:", Total_n, "\n")
prop.test(Blond_n, Total_n, alternative = "less")

In [ ]:
# 2x4 homogeneity: weight category distribution by sex
cont_hom2 <- matrix(
    c( 80, 160, 120,  40,
       60, 180, 240, 120),
    nrow = 2, ncol = 4, byrow = TRUE
)
colnames(cont_hom2) <- c("<30kg", "30-50kg", "50-100kg", ">100kg")
rownames(cont_hom2) <- c("Female", "Male")

chisq.test(cont_hom2)

---

## 6. Linear Regression

Linear regression models the relationship between a response variable $Y$ and one or more predictors $X$:

$$Y = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2)$$

The `lm` function fits the model by ordinary least squares. After fitting, always check residuals for normality — this is an assumption of inference, not of estimation.

In [ ]:
# Boston housing dataset: predict median home value (medv) from
# percentage of lower-status population (lstat)
Boston_data <- read.table("../Assets/data/r_statistics/Boston.csv",
                          header = TRUE, sep = ";", dec = ",")

medv  <- Boston_data$medv
lstat <- Boston_data$lstat

# Fit simple linear regression
boston_lm <- lm(medv ~ lstat)
summary(boston_lm)

In [ ]:
# Coefficient estimates and confidence intervals
coef(boston_lm)
confint(boston_lm)

In [ ]:
# Scatter plot with regression line
plot(lstat, medv, pch = 20,
     xlab = "% Lower-Status Population (lstat)",
     ylab = "Median Home Value (medv, $1000s)",
     main = "Boston Housing: Simple Linear Regression")

abline(boston_lm, lwd = 2, col = "red")

# Alternative: plot fitted values in sorted order
z <- order(lstat)
lines(lstat[z], boston_lm$fitted.values[z], col = "blue", lwd = 2, lty = 2)

### 6.1 Residual Diagnostics

The t-tests and F-tests in `summary(lm(...))` are only valid when residuals are approximately normal. Always check this assumption.

In [ ]:
# Extract residuals
boston_res <- boston_lm$residuals

par(mfrow = c(1, 2))
hist(boston_res, breaks = 18,
     main = "Residual Distribution",
     xlab = "Residual", col = "lightgrey")
qqnorm(boston_res, main = "Q-Q Plot of Residuals")
qqline(boston_res, col = "red")
par(mfrow = c(1, 1))

# Shapiro-Wilk on residuals
shapiro.test(boston_res)

### 6.2 Polynomial Regression

If the scatter plot suggests a curved relationship, polynomial terms can be added.

In [ ]:
# Quadratic fit
lstat_2     <- lstat^2
boston_lm_2 <- lm(medv ~ lstat + lstat_2)
summary(boston_lm_2)

# Plot quadratic fit
plot(lstat, medv, pch = 20,
     xlab = "lstat", ylab = "medv",
     main = "Quadratic Regression")
z <- order(lstat)
lines(lstat[z], boston_lm_2$fitted.values[z], col = "blue", lwd = 2)

In [ ]:
# Degree-7 polynomial using poly() — orthogonal polynomial basis
boston_lm_poly <- lm(medv ~ poly(lstat, 7))
summary(boston_lm_poly)

# Plot
plot(lstat, medv, pch = 20,
     xlab = "lstat", ylab = "medv",
     main = "Degree-7 Polynomial Regression")
z <- order(lstat)
lines(lstat[z], boston_lm_poly$fitted.values[z], col = "blue", lwd = 2)

### 6.3 Multiple Linear Regression

In [ ]:
# Multiple regression: medv ~ lstat + age + rm + crim
age   <- Boston_data$age
rm    <- Boston_data$rm
crim  <- Boston_data$crim

boston_lm_multi <- lm(medv ~ lstat + age + rm + crim)
summary(boston_lm_multi)

In [ ]:
# Second dataset: car seat sales ~ price + advertising + age + population
# (Carseats data from ISLR)
# We use Nurses_1.csv as an alternative regression example
nurses_data  <- read.table("../Assets/data/r_statistics/Nurses_1.csv",
                           header = TRUE, sep = ";", dec = ",")
head(nurses_data)
str(nurses_data)

---

## 7. ANOVA vs Kruskal-Wallis

One-way **ANOVA** partitions total variance into between-group and within-group components. It assumes normally distributed residuals with equal variances.  
**Kruskal-Wallis** is the rank-based alternative that relaxes normality.  

When data are normal, ANOVA is more powerful. When not, Kruskal-Wallis is safer.

In [ ]:
# Pharmacy revenue by store number
# H0: mean (ANOVA) / median (K-W) revenue is equal across stores
Pharmacy_data   <- read.table("../Assets/data/r_statistics/Pharmacy_1.csv",
                              header = TRUE, sep = ";", dec = ",")
Pharmacy_number <- Pharmacy_data[[1]]
Pharmacy_money  <- Pharmacy_data[[2]]

cat("Groups (stores):", length(unique(Pharmacy_number)), "\n")
cat("Total observations:", length(Pharmacy_money), "\n")

In [ ]:
# ANOVA via lm + anova()
lm_result <- lm(Pharmacy_money ~ Pharmacy_number)
anova(lm_result)

In [ ]:
# Kruskal-Wallis on the same data for comparison
kruskal.test(Pharmacy_money, Pharmacy_number)

In [ ]:
# Check ANOVA residuals for normality
anova_res <- lm_result$residuals
shapiro.test(anova_res)

par(mfrow = c(1, 2))
hist(anova_res, breaks = 12,
     main = "ANOVA Residuals", xlab = "Residual", col = "lightyellow")
qqnorm(anova_res); qqline(anova_res, col = "red")
par(mfrow = c(1, 1))

---

## 8. Law of Large Numbers and Central Limit Theorem

### 8.1 CLT: Sample Means Converge to Normal

The Central Limit Theorem (CLT) states that for large $n$, the distribution of the sample mean $\bar{X}_n$ is approximately $N(\mu, \sigma^2/n)$, regardless of the original distribution.

In [ ]:
# Draw N=10000 samples of size n=8 from Uniform[0,1]
set.seed(42)
n <- 8
N <- 10000

M <- matrix(ncol = n, nrow = N, data = runif(n * N))

# Single draw from the original distribution
hist(M[, 1], main = "Single Observation (Uniform[0,1])",
     xlab = "Value", col = "lightblue", breaks = 20)

In [ ]:
# Distribution of sample means with n=8
x_bar <- apply(M, 1, mean)   # row-wise means
hist(x_bar, main = "Distribution of Sample Mean (n=8)",
     xlab = expression(bar(X)), col = "lightgreen", breaks = 50)

In [ ]:
# CLT: variance of sampling distribution decreases as n increases
par(mfrow = c(3, 1))
for (n_size in c(8, 25, 100)) {
    M_n <- matrix(ncol = n_size, nrow = N, data = runif(n_size * N))
    x_n <- apply(M_n, 1, mean)
    hist(x_n, xlim = c(0, 1), breaks = 50,
         main = paste0("n = ", n_size),
         xlab = expression(bar(X)), col = "lightblue")
}
par(mfrow = c(1, 1))

### 8.2 Law of Large Numbers: SD of Sample Mean ∝ 1/√n

In [ ]:
# Empirical standard deviation of sample means vs theoretical 1/sqrt(n)
sizes <- c(8, 16, 30, 50, 100, 200, 400)
s_emp <- c()

for (n_size in sizes) {
    M_n   <- matrix(ncol = n_size, nrow = N, data = runif(n_size * N))
    x_n   <- apply(M_n, 1, mean)
    s_emp <- c(s_emp, sd(x_n))
}

par(mfrow = c(1, 2))
plot(sizes, s_emp,
     xlab = "n", ylab = "sd(x_bar)",
     main = "SD of Sample Mean", pch = 20, col = "steelblue")

# Log-log scale: should show slope = -0.5
plot(sizes, s_emp, log = "xy",
     xlab = "log(n)", ylab = "log(sd(x_bar))",
     main = "Log-Log Scale", pch = 20, col = "steelblue")
par(mfrow = c(1, 1))

In [ ]:
# Test whether the slope in log-log regression equals -0.5
df_lln  <- data.frame(n = sizes, s = s_emp)
model   <- lm(log(s) ~ log(n), data = df_lln)
summary(model)

b  <- summary(model)$coefficients[2, 1]   # slope estimate
SE <- summary(model)$coefficients[2, 2]   # standard error

# t-test: H0: slope = -0.5
t_val   <- abs(b + 0.5) / SE
p_value <- pt(t_val, df = length(sizes) - 2)
cat(sprintf("Slope estimate: %.4f  SE: %.4f\n", b, SE))
cat(sprintf("Test H0: slope = -0.5  p-value: %.4f\n", p_value))

### 8.3 Confidence Interval Coverage Simulation

A 95% CI should contain the true parameter in 95% of repeated experiments. This simulation demonstrates why we must use the t-quantile (not the z-quantile) when $\sigma$ is unknown.

In [ ]:
# True parameters for a normal population
mu    <- 41.6
sigma <- 6.05
n     <- 17
N     <- 10000

set.seed(42)
M_ci  <- matrix(ncol = n, nrow = N, data = rnorm(n * N, mu, sigma))
df_ci <- data.frame(
    xbar = apply(M_ci, 1, mean),
    sx   = apply(M_ci, 1, sd)
)

In [ ]:
# CI using z-quantile (WRONG — sigma is unknown)
df_ci$ME_z    <- qnorm(0.025, lower.tail = FALSE) * df_ci$sx / sqrt(n)
df_ci$left_z  <- df_ci$xbar - df_ci$ME_z
df_ci$right_z <- df_ci$xbar + df_ci$ME_z
df_ci$hit_z   <- df_ci$left_z < mu & mu < df_ci$right_z

p_hat_z <- table(df_ci$hit_z)["TRUE"] / N
cat(sprintf("Coverage with z-quantile: %.3f (target 0.95)\n", p_hat_z))

# One-proportion test: H0: coverage = 0.95
z_cov <- (p_hat_z - 0.95) / sqrt(0.95 * 0.05 / N)
cat(sprintf("p-value: %.4f — H0 rejected: %s\n", pnorm(z_cov), pnorm(z_cov) < 0.05))

In [ ]:
# CI using t-quantile (CORRECT)
df_ci$ME_t    <- qt(0.025, df = n - 1, lower.tail = FALSE) * df_ci$sx / sqrt(n)
df_ci$left_t  <- df_ci$xbar - df_ci$ME_t
df_ci$right_t <- df_ci$xbar + df_ci$ME_t
df_ci$hit_t   <- df_ci$left_t < mu & mu < df_ci$right_t

p_hat_t <- table(df_ci$hit_t)["TRUE"] / N
cat(sprintf("Coverage with t-quantile: %.3f (target 0.95)\n", p_hat_t))

z_cov_t <- (p_hat_t - 0.95) / sqrt(0.95 * 0.05 / N)
cat(sprintf("p-value: %.4f — H0 not rejected (correct)\n", pnorm(z_cov_t)))

In [ ]:
# CI using z-quantile but with TRUE sigma (also correct)
df_ci$ME_zt   <- qnorm(0.025, lower.tail = FALSE) * sigma / sqrt(n)
df_ci$left_zt  <- df_ci$xbar - df_ci$ME_zt
df_ci$right_zt <- df_ci$xbar + df_ci$ME_zt
df_ci$hit_zt   <- df_ci$left_zt < mu & mu < df_ci$right_zt

p_hat_zt <- table(df_ci$hit_zt)["TRUE"] / N
cat(sprintf("Coverage with z-quantile + known sigma: %.3f\n", p_hat_zt))

The coverage simulation confirms:
- Using a z-quantile when $\sigma$ is **unknown** undercovers (< 95%).
- Using a t-quantile achieves the nominal coverage.
- Using a z-quantile when $\sigma$ is **known** also achieves nominal coverage.

---

## 9. Exercises

**Exercise 1 — Confidence Intervals**  
Load `IQ.txt`. Construct a 95% two-sided Student-t CI for the mean IQ. Also test $H_0: \mu = 110$ against $H_1: \mu > 110$ at $\alpha = 0.05$ using both manual computation and `t.test`.

**Exercise 2 — Parametric t-Tests**  
Load `Weight.txt`. First run the paired t-test ($H_1$: after < before). Then load `town.txt` and run an independent two-sample test: (a) use `var.test` to check variance equality, (b) apply the appropriate t-test variant.

**Exercise 3 — Normality Testing**  
Load `weather.txt`. Extract temperatures for July only. Apply Shapiro-Wilk, Pearson chi-squared (with and without adjustment), and produce a Q-Q plot. Based on the results, is the t-test appropriate for July temperatures?

**Exercise 4 — Correlation**  
Load `country.txt`. Compute Pearson, Spearman, and Kendall correlations between the two numeric columns. Manually verify the Pearson r and its Fisher z-transform CI. Compare the three correlation coefficients and explain any differences.

**Exercise 5 — Contingency Tables**  
Build the 2×4 response-time × age-group table from Section 5.1 and run the chi-squared independence test. Then use `students.txt` to test whether the distribution of hair color is homogeneous between male and female students.

**Exercise 6 — Regression Diagnostics**  
Load `Boston.csv`. Fit a simple regression of `medv` on `lstat`. Check residual normality with `shapiro.test` and a Q-Q plot. Fit a quadratic model and compare $R^2$ values. Explain whether the polynomial improvement is meaningful.

In [ ]:
# Exercise 1 workspace


In [ ]:
# Exercise 2 workspace


In [ ]:
# Exercise 3 workspace


In [ ]:
# Exercise 4 workspace


In [ ]:
# Exercise 5 workspace


In [ ]:
# Exercise 6 workspace


---

## Summary

| Topic | Key Functions |
|-------|---------------|
| Student-t CI | `qt`, `t.test` |
| Chi-squared CI for variance | `qchisq` |
| Normal asymptotic CI | `qnorm` |
| Paired t-test | `t.test(paired=TRUE)` |
| Independent t-test | `t.test(paired=FALSE, var.equal=?)` |
| F-test for equal variances | `var.test` |
| Shapiro-Wilk normality | `shapiro.test` |
| Pearson chi-squared GoF | `pearson.test` (nortest) |
| Q-Q plot | `qqnorm` + `qqline` |
| Pearson / Spearman / Kendall | `cor.test(..., method=)` |
| Fisher z CI | manual: `0.5*log((1+r)/(1-r))` |
| Chi-squared independence | `chisq.test` |
| Proportion homogeneity | `prop.test` |
| Contingency table | `xtabs` + `ftable` |
| Linear regression | `lm`, `summary`, `confint` |
| Polynomial regression | `lm(y ~ poly(x, deg))` |
| ANOVA | `lm` + `anova` |
| CLT / LLN simulation | `matrix`, `apply`, `hist` |

**Decision rule for t-test vs Wilcoxon:**
1. Check normality with `shapiro.test`.
2. If p > 0.05 (no evidence against normality): use t-test.
3. If p < 0.05 (normality violated): use Wilcoxon / Kruskal-Wallis.
4. For independent samples: always run `var.test` first to choose equal vs Welch variant.

---

[< Previous: R Hypothesis Testing and Nonparametric Methods](01_r_hypothesis_testing_and_nonparametrics.ipynb) | [Tier 0: Computational Foundations Overview](../README.md)

---